# CCAR deposit NII & margin — full-path versus full-horizon aggregate Shapley

**Inputs:** FTP dollars, interest-expense dollars, and average total/DDA/IB balances per month or quarter. **Outputs:** two mathematically exact four-factor decompositions of the *same final cumulative NII and margin*: (1) full-period path Shapley and (2) aggregate-only Shapley. The allocations can differ because full-path analysis captures balance timing and correlations across periods. Both reconcile to the reporting totals.

**Scope:** CCAR Baseline vs Business Outlook and change in stress-minus-baseline across annual CCAR cycles. This is an accounting allocation, not causal identification. The blended FTP-yield effect can include FTP-related product mix changes.

## 1. User controls and simulated inputs

Set frequency to `Q` (CSV labels `PQ0`, `PQ1`, ...; `Q0` is also accepted) or `M` (`M0`, `M1`, ...). `N_PERIODS` counts *included rows*: Q1–Q9 is 9; PQ0–PQ8 is 9. `INCLUDE_INITIAL=True` includes period 0 **as a full flow period**, only if the Q0/M0 FTP and expense values really are full-period flows. Otherwise exclude Q0.

**Margin conventions:** `reported` = cumulative NII / arithmetic mean of the selected period-average balances (no annualization). `annualized` = cumulative NII / (average balance × selected years). Both are calculated and exported. Choose the convention that matches your P&R report. Inputs and all computations remain **in dollars**; charts display millions.

In [ ]:
import itertools
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# ========== CHANGE ONLY THESE SETTINGS ==========
FREQUENCY = "Q"        # "Q" or "M"
N_PERIODS = 9          # Number of included observations (e.g., 8 or 9 quarters; 24/27/28 months)
INCLUDE_INITIAL = False # False: Q1..Q9; True: Q0..Q8 (when N_PERIODS=9)
MARGIN_CONVENTION = "reported"   # "reported" (NII / avg balance) or "annualized"
DATA_START_INDEX = 0   # For dated files: 0 if first row is Q0/M0, otherwise 1
CSV_PATH = None        # None = simulate; or r"C:\path\my_data.csv"
CURRENT_CYCLE = "2026"
PRIOR_CYCLE = "2025"
BASE = "Baseline"
STRESS = "Severe"
OUTLOOK = "Outlook"

# CSV column names -- change right-hand values for your actual file
COLUMN_MAP = {
    'period': ('Quarter' if FREQUENCY == 'Q' else 'Month'), 'scenario': 'Scenario', 'cycle': 'Cycle',
    'ftp': 'FTP', 'expense': 'Interest_Expense',
    'total': 'Total_Deposit', 'dda': 'DDA', 'ib': 'IB',
    'fed_funds': 'Fed_Funds',
}
# =================================================
FREQUENCY = FREQUENCY.upper().strip()
if MARGIN_CONVENTION not in ("reported", "annualized"):
    raise ValueError("MARGIN_CONVENTION must be reported or annualized")
if FREQUENCY not in ("Q", "M"):
    raise ValueError("FREQUENCY must be Q or M")
if not isinstance(N_PERIODS, int) or N_PERIODS < 1:
    raise ValueError("N_PERIODS must be a positive integer")
PERIODS_PER_YEAR = 4 if FREQUENCY == "Q" else 12
TAU = 1 / PERIODS_PER_YEAR
FIRST_INCLUDED = 0 if INCLUDE_INITIAL else 1
PERIOD_IDS = list(range(FIRST_INCLUDED, FIRST_INCLUDED + N_PERIODS))
OUTPUT = Path('deposit_notebook_results')
OUTPUT.mkdir(parents=True, exist_ok=True)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
print(f"Using {FREQUENCY}: {PERIOD_IDS[0]}..{PERIOD_IDS[-1]} ({N_PERIODS/PERIODS_PER_YEAR:g} years)")
HORIZON_YEARS = N_PERIODS / PERIODS_PER_YEAR
MARGIN_MULTIPLIER = HORIZON_YEARS if MARGIN_CONVENTION == "reported" else 1.0
MARGIN_UNITS = ("reported, nonannualized bps" if MARGIN_CONVENTION == "reported" else "annualized bps")
print("Margin convention:", MARGIN_CONVENTION, "; exposure years:", HORIZON_YEARS)


In [ ]:
# Simulate only when CSV_PATH is None. Your actual data needs only raw dollar fields.
def make_sample():
    rng = np.random.default_rng(20261109)
    rows=[]
    max_i=max(PERIOD_IDS)
    for cycle, book_scale in [('2025',1.0),('2026',1.075)]:
        for scenario in ['Baseline','Severe','Outlook']:
            for t in range(max_i+1):
                total = 108_000_000_000 * book_scale * (1+0.002*t) * {
                    'Baseline':1.0,'Severe':0.96-0.0012*t,'Outlook':1.025+0.0005*t}[scenario]
                w = {'Baseline':0.385,'Severe':0.355,'Outlook':0.395}[scenario]-0.0006*t
                dda=total*w; ib=total-dda
                ff={'Baseline':4.3-0.045*t,'Severe':4.3-0.095*t,'Outlook':4.2-0.040*t}[scenario]
                ff += 0.06 if cycle=='2026' else 0
                f=max(0.004,ff/100-0.0025+(0.0006 if scenario=='Outlook' else 0))
                r=max(0.002,0.013+0.43*(ff/100-0.03))
                rows.append([f'{"PQ" if FREQUENCY == "Q" else "M"}{t}',scenario,cycle,total*f*TAU,ib*r*TAU,total,dda,ib,ff])
    return pd.DataFrame(rows,columns=['Quarter' if FREQUENCY=='Q' else 'Month', 'Scenario', 'Cycle',
                                      'FTP','Interest_Expense','Total_Deposit','DDA','IB','Fed_Funds'])

if CSV_PATH is None:
    simulated = make_sample()
    csv_path = OUTPUT / f'simulated_{FREQUENCY}.csv'
    simulated.to_csv(csv_path,index=False)
    print('Using simulated CSV:',csv_path)
    display(simulated.head())
else:
    csv_path = Path(CSV_PATH)
    print('Using actual CSV:',csv_path)

## 2. Core calculation and exact Shapley engine

Period-by-period: `NII_t = FTP_t − InterestExpense_t = B_t × τ × [f_t − (1−w_t)r_t]`, where `w_t = DDA_t / B_t`, `f_t = FTP_t / (B_t τ)`, and `r_t = Expense_t / (IB_t τ)`.

**Path method:** For each of the 24 factor-switching orders, replace an entire factor *series*; recalculate **cumulative NII and the full-horizon ratio**. Do **not** average quarterly margin contributions to get the final margin.

**Aggregate method:** First collapse the whole horizon to total FTP, total expense, average balances, DDA share, and implied *full-horizon* FTP and paid yields. Switch these **aggregate** factors using the same Shapley engine. Totals are identical under both methods, but driver allocations need not be.

In [ ]:
import itertools
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FACTORS = ('Balance', 'Mix', 'FTP yield', 'IB paid rate')
COLORS = ['#3274A1','#9064A1','#C48736','#32866C']


def load_data(path, mapping=None):
    mapping = COLUMN_MAP if mapping is None else mapping
    raw = pd.read_csv(path)
    mandatory = ['period','scenario','ftp','expense','total','dda','ib']
    missing = [mapping[k] for k in mandatory if mapping[k] not in raw.columns]
    if missing: raise ValueError(f'Missing required columns: {missing}')
    optional = [k for k in ('cycle','fed_funds') if mapping.get(k) in raw.columns]
    names = mandatory + optional
    x = raw[[mapping[k] for k in names]].rename(columns={mapping[k]:k for k in names}).copy()
    if 'cycle' not in x.columns: x['cycle']='single'
    x['cycle'] = x['cycle'].astype(str)
    x['scenario'] = x['scenario'].astype(str)
    for k in ('ftp','expense','total','dda','ib'):
        x[k] = pd.to_numeric(x[k],errors='raise')
    if x[['ftp','expense','total','dda','ib']].isna().any().any():
        raise ValueError('Missing numeric values')
    if (x[['total','ib']] <= 0).any().any() or (x.dda < 0).any():
        raise ValueError('Total and IB must be positive; DDA nonnegative')
    if (x.expense < 0).any(): raise ValueError('Interest expense must be a positive dollar cost')
    if ((x.total-x.dda-x.ib).abs() > np.maximum(1e-6,x.total.abs()*1e-6)).any():
        raise ValueError('Total average balance must equal DDA + IB')

    # Accept Q0, Q1... or M0, M1... directly; otherwise accept dates.
    labels = x['period'].astype(str).str.strip().str.upper()
    match = labels.str.extract(r'^(PQ|Q|M)(\d+)$')
    if match.notna().all().all():
        valid_prefixes = ('PQ', 'Q') if FREQUENCY == 'Q' else ('M',)
        if (~match[0].isin(valid_prefixes)).any():
            raise ValueError('Period labels do not match FREQUENCY')
        x['period_index'] = match[1].astype(int)
        x['calendar_date'] = pd.NaT
    else:
        x['calendar_date'] = pd.to_datetime(x['period'],errors='raise')
        x = x.sort_values(['cycle','scenario','calendar_date'])
        x['period_index'] = x.groupby(['cycle','scenario']).cumcount() + DATA_START_INDEX

    key=['cycle','scenario','period_index']
    if x.duplicated(key).any(): raise ValueError('Duplicate cycle/scenario/period rows')
    x=x[x.period_index.isin(PERIOD_IDS)].copy().sort_values(key).reset_index(drop=True)
    if x.empty: raise ValueError('No rows match the chosen horizon')
    for (cycle,scenario),g in x.groupby(['cycle','scenario'],sort=False):
        if g.period_index.tolist() != PERIOD_IDS:
            raise ValueError(f'{cycle}/{scenario}: expected {PERIOD_IDS}; got {g.period_index.tolist()}')
    x['tau'] = TAU
    x['mix'] = x.dda/x.total
    x['ftp_yield'] = x.ftp/(x.total*x.tau)
    x['paid_yield'] = x.expense/(x.ib*x.tau)
    x['nii'] = x.ftp-x.expense
    return x

def factor_arrays(d):
    return {'Balance':d.total.to_numpy(float), 'Mix':d.mix.to_numpy(float),
            'FTP yield':d.ftp_yield.to_numpy(float),
            'IB paid rate':d.paid_yield.to_numpy(float)}


def evaluate(state,tau):
    b=state['Balance']; w=state['Mix']; f=state['FTP yield']; r=state['IB paid rate']
    exposure=b*tau
    monthly=exposure*(f-(1-w)*r)
    return monthly, float(monthly.sum()/exposure.sum())


def shapley(A,B,tau):
    n={k:np.zeros(len(tau)) for k in FACTORS}
    m={k:0.0 for k in FACTORS}
    for order in itertools.permutations(FACTORS):
        state=dict(A)
        prev_nii,prev_margin=evaluate(state,tau)
        for k in order:
            state[k]=B[k]
            next_nii,next_margin=evaluate(state,tau)
            n[k]+=(next_nii-prev_nii)/24
            m[k]+=(next_margin-prev_margin)*10000/24
            prev_nii,prev_margin=next_nii,next_margin
    return n,m




def aggregate_factors(d):
    """Exact full-horizon compression (not a simple average of paid yields).

    Bbar = sum(tau*B)/T, wbar = sum(tau*DDA)/sum(tau*B),
    fbar = sum(FTP)/sum(tau*B), rbar = sum(Expense)/sum(tau*IB).
    One synthetic period has tau=T, so evaluate() exactly reproduces
    cumulative NII and the exposure-weighted annualized margin.
    """
    exposure = float((d.total * d.tau).sum())
    ib_exposure = float((d.ib * d.tau).sum())
    T = float(d.tau.sum())
    if exposure <= 0 or ib_exposure <= 0 or T <= 0:
        raise ValueError("Invalid aggregate deposit exposure")
    return {
        'Balance':np.array([exposure/T]),
        'Mix':np.array([float((d.dda*d.tau).sum()/exposure)]),
        'FTP yield':np.array([float(d.ftp.sum()/exposure)]),
        'IB paid rate':np.array([float(d.expense.sum()/ib_exposure)]),
    }, np.array([T], dtype=float)


def compare_aggregate(a,b):
    """Aggregate Shapley: exact same report totals; different factor allocation."""
    A, tau = aggregate_factors(a)
    B, tau_b = aggregate_factors(b)
    if not np.allclose(tau,tau_b):
        raise ValueError('Horizon lengths differ')
    na,ma=evaluate(A,tau);nb,mb=evaluate(B,tau)
    contributions_n,contributions_m=shapley(A,B,tau)
    result={
        'nii_a':float(na.sum()), 'nii_b':float(nb.sum()),
        'margin_a':ma*10000,'margin_b':mb*10000,
        'n':{k:float(contributions_n[k].sum()) for k in FACTORS},
        'm':contributions_m,
    }
    if not np.isclose(sum(result['n'].values()),result['nii_b']-result['nii_a'],rtol=1e-11,atol=1e-3):
        raise AssertionError('Aggregate NII attribution does not reconcile')
    if not np.isclose(sum(result['m'].values()),result['margin_b']-result['margin_a'],rtol=0,atol=1e-7):
        raise AssertionError('Aggregate margin attribution does not reconcile')
    return result


def report_summary(d):
    ftp=float(d.ftp.sum()); expense=float(d.expense.sum())
    nii=ftp-expense
    mean_balance=float(d.total.mean())
    total_years=float(d.tau.sum())
    exposure=float((d.total*d.tau).sum())
    return {
        'Cumulative FTP ($)':ftp,
        'Cumulative expense ($)':expense,
        'Cumulative NII ($)':nii,
        'Average balance ($)':mean_balance,
        'Average DDA ($)':float(d.dda.mean()),
        'Average IB ($)':float(d.ib.mean()),
        'DDA share (%)':100*float(d.dda.sum()/d.total.sum()),
        'FTP implied annual yield (%)':100*ftp/exposure,
        'IB paid implied annual yield (%)':100*expense/float((d.ib*d.tau).sum()),
        'Reported margin (bps)':10000*nii/mean_balance,
        'Annualized margin (bps)':10000*nii/exposure,
        'Horizon (years)':total_years,
    }


def chosen_margin(result):
    return (result['margin_a']*MARGIN_MULTIPLIER,
            result['margin_b']*MARGIN_MULTIPLIER,
            {k:v*MARGIN_MULTIPLIER for k,v in result['m'].items()})


def selected(x,cycle,scenario):
    d=x[(x.cycle==str(cycle))&(x.scenario==scenario)].sort_values('period_index').reset_index(drop=True)
    if d.empty: raise ValueError(f'No data for cycle={cycle}, scenario={scenario}')
    return d


def compare(a,b,allow_calendar_shift=False):
    if len(a)!=len(b): raise ValueError('Both paths must have the same number of observations')
    if not a.period_index.equals(b.period_index):
        raise ValueError('Comparison requires matching projection indices')
    if not allow_calendar_shift and not a.period.equals(b.period):
        raise ValueError('Comparison requires matching period labels/dates')
    if not np.allclose(a.tau.to_numpy(),b.tau.to_numpy()):
        raise ValueError('Accrual fractions differ')
    if len(a)!=N_PERIODS: raise ValueError('Unexpected number of periods')
    tau=a.tau.to_numpy()
    A=factor_arrays(a);B=factor_arrays(b)
    na,ma=evaluate(A,tau); nb,mb=evaluate(B,tau)
    contributions_n,contributions_m=shapley(A,B,tau)
    result={
        'nii_a':float(na.sum()),'nii_b':float(nb.sum()),
        'margin_a':ma*10000,'margin_b':mb*10000,
        'n':{k:float(contributions_n[k].sum()) for k in FACTORS},
        'm':contributions_m,
        'monthly':pd.DataFrame({'month_index':a.period_index.to_numpy(),'nii_a':na,'nii_b':nb,
                                **{f'{k}_nii':contributions_n[k] for k in FACTORS}}),
        'exposure_a':float((a.total*a.tau).sum()),
        'exposure_b':float((b.total*b.tau).sum()),
    }
    assert np.isclose(sum(result['n'].values()),result['nii_b']-result['nii_a'],atol=1e-3,rtol=0)
    assert np.isclose(sum(result['m'].values()),result['margin_b']-result['margin_a'],atol=1e-8,rtol=0)
    assert np.allclose(result['monthly'][[f'{k}_nii' for k in FACTORS]].sum(axis=1),nb-na,atol=1e-3)
    return result


def plot_waterfall(path,title,start,contributions,unit):
    keys=list(contributions)
    values=np.array([contributions[k] for k in keys])
    fig,ax=plt.subplots(figsize=(10.5,4.7))
    ax.bar(0,start,color='#30485e',width=.68)
    current=start
    for i,(k,v) in enumerate(zip(keys,values),start=1):
        ax.bar(i,abs(v),bottom=min(current,current+v),color=COLORS[i-1],width=.68)
        ax.text(i,current+v/2,f'{v:+,.1f}',ha='center',va='center',fontsize=9,fontweight='bold')
        current+=v
    ax.bar(len(keys)+1,current,color='#30485e',width=.68)
    ax.set_xticks(range(len(keys)+2),['Reference',*keys,'Comparison'])
    ax.set_ylabel(unit)
    ax.set_title(title,loc='left',fontweight='bold',fontsize=14)
    ax.axhline(0,lw=.7,color='gray');ax.grid(axis='y',alpha=.12)
    ax.spines[['top','right']].set_visible(False)
    fig.tight_layout();fig.savefig(path,dpi=170,bbox_inches='tight');plt.close(fig)


def save_result(out,prefix,result,description):
    tab=pd.DataFrame({'factor':FACTORS,'nii_dollars':[result['n'][k] for k in FACTORS],
                      'margin_bps':[result['m'][k] for k in FACTORS]})
    tab.to_csv(out/f'{prefix}_attribution.csv',index=False)
    result['monthly'].to_csv(out/f'{prefix}_monthly.csv',index=False)
    plot_waterfall(out/f'{prefix}_nii.png',description+' — NII',result['nii_a'],result['n'],'NII ($ input units)')
    plot_waterfall(out/f'{prefix}_margin.png',description+' — margin',result['margin_a'],result['m'],'Annualized bps')
    return tab




## 3. Load data; compare CCAR Baseline vs Business Outlook

Both methods *reconstruct the identical reporting totals*. Display the two allocations side-by-side to see whether balance timing changes the allocation. The main executive waterfalls use **full-horizon aggregate Shapley**; the path method remains available as a sensitivity check.

In [ ]:
data = load_data(csv_path, mapping=COLUMN_MAP)
print('Rows:',len(data),'selected periods:',PERIOD_IDS)
display(data.head())

a = selected(data,CURRENT_CYCLE,BASE)
b = selected(data,CURRENT_CYCLE,OUTLOOK)
pair_path = compare(a,b)          # Full quarterly/monthly path
pair_agg = compare_aggregate(a,b) # One full-horizon aggregate observation

# Independent direct accounting verification
sa, sb = report_summary(a), report_summary(b)
for label, r in [('Path',pair_path),('Aggregate',pair_agg)]:
    assert np.isclose(r['nii_a'],sa['Cumulative NII ($)'],rtol=1e-11,atol=1e-3)
    assert np.isclose(r['nii_b'],sb['Cumulative NII ($)'],rtol=1e-11,atol=1e-3)
    assert np.isclose(r['margin_a'],sa['Annualized margin (bps)'],atol=1e-7)
    assert np.isclose(r['margin_b'],sb['Annualized margin (bps)'],atol=1e-7)

print('FINAL REPORTED TOTALS (A = CCAR Base, B = Outlook)')
display(pd.DataFrame([sa,sb],index=['CCAR Base','Business Outlook']).T)

comparison = pd.DataFrame({
    'Driver':FACTORS,
    'Path NII ($MM)':[pair_path['n'][f]/1e6 for f in FACTORS],
    'Aggregate NII ($MM)':[pair_agg['n'][f]/1e6 for f in FACTORS],
    f'Path margin ({MARGIN_UNITS})':[pair_path['m'][f]*MARGIN_MULTIPLIER for f in FACTORS],
    f'Aggregate margin ({MARGIN_UNITS})':[pair_agg['m'][f]*MARGIN_MULTIPLIER for f in FACTORS],
})
display(comparison)
comparison.to_csv(OUTPUT/'pair_path_vs_aggregate.csv',index=False)

# Main executive waterfalls: full-horizon aggregate
ma,mb,mm = chosen_margin(pair_agg)
plot_waterfall(OUTPUT/'01_base_vs_outlook_nii.png', 'Base vs Outlook — full-horizon NII',
               pair_agg['nii_a']/1e6,{k:v/1e6 for k,v in pair_agg['n'].items()},'$ millions')
plot_waterfall(OUTPUT/'02_base_vs_outlook_margin.png','Base vs Outlook — full-horizon margin',
               ma,mm,MARGIN_UNITS)
print('Aggregate NII bridge:',pair_agg['nii_a']/1e6,'→',pair_agg['nii_b']/1e6,'$MM')
print('Aggregate margin bridge:',round(ma,3),'→',round(mb,3),MARGIN_UNITS)


## 4. Cycle-over-cycle: change in (Stress − Baseline)

Calculate the stress impact for each cycle separately, then subtract **current minus prior** driver allocations. Both path and aggregate versions reconcile to the exact difference-in-differences, in dollars and the selected margin convention.

In [ ]:
prior_base=selected(data,PRIOR_CYCLE,BASE)
prior_stress=selected(data,PRIOR_CYCLE,STRESS)
current_base=selected(data,CURRENT_CYCLE,BASE)
current_stress=selected(data,CURRENT_CYCLE,STRESS)

prior_path=compare(prior_base,prior_stress)
current_path=compare(current_base,current_stress)
prior_agg=compare_aggregate(prior_base,prior_stress)
current_agg=compare_aggregate(current_base,current_stress)

cycle_rows=[]
for k in FACTORS:
    cycle_rows.append({
        'Driver':k,
        'Prior stress NII ($MM)':prior_agg['n'][k]/1e6,
        'Current stress NII ($MM)':current_agg['n'][k]/1e6,
        'Cycle change: aggregate NII ($MM)':(current_agg['n'][k]-prior_agg['n'][k])/1e6,
        'Cycle change: path NII ($MM)':(current_path['n'][k]-prior_path['n'][k])/1e6,
        f'Cycle change: aggregate margin ({MARGIN_UNITS})':(current_agg['m'][k]-prior_agg['m'][k])*MARGIN_MULTIPLIER,
        f'Cycle change: path margin ({MARGIN_UNITS})':(current_path['m'][k]-prior_path['m'][k])*MARGIN_MULTIPLIER,
    })
cycle_table=pd.DataFrame(cycle_rows)
display(cycle_table)
cycle_table.to_csv(OUTPUT/'cycle_path_vs_aggregate.csv',index=False)

prior_stress_nii=prior_agg['nii_b']-prior_agg['nii_a']
current_stress_nii=current_agg['nii_b']-current_agg['nii_a']
prior_stress_margin=(prior_agg['margin_b']-prior_agg['margin_a'])*MARGIN_MULTIPLIER
current_stress_margin=(current_agg['margin_b']-current_agg['margin_a'])*MARGIN_MULTIPLIER
cycle_nii={k:current_agg['n'][k]-prior_agg['n'][k] for k in FACTORS}
cycle_margin={k:(current_agg['m'][k]-prior_agg['m'][k])*MARGIN_MULTIPLIER for k in FACTORS}

plot_waterfall(OUTPUT/'03_cycle_change_nii.png','Cycle change in stress NII impact',
               prior_stress_nii/1e6,{k:v/1e6 for k,v in cycle_nii.items()},'$ millions')
plot_waterfall(OUTPUT/'04_cycle_change_margin.png','Cycle change in stress margin impact',
               prior_stress_margin,cycle_margin,MARGIN_UNITS)
print('Prior → current stress NII impact ($MM):',round(prior_stress_nii/1e6,3),'→',round(current_stress_nii/1e6,3))
print('Prior → current stress margin impact (bps):',round(prior_stress_margin,3),'→',round(current_stress_margin,3))


## 5. Reporting check: totals, balance exposure and normalized stress impact

`Reported margin = cumulative NII / mean balance`; `Annualized margin = cumulative NII / (mean balance × number of years)`. The normalized stress NII impact divides the stress-minus-base NII gap by the **base exposure** and is not the same as the stress-minus-base margin gap.

In [ ]:
reporting=pd.DataFrame({
    'Current CCAR Base':report_summary(a),
    'Current Business Outlook':report_summary(b),
    'Prior CCAR Base':report_summary(prior_base),
    'Prior Stress':report_summary(prior_stress),
    'Current Stress':report_summary(current_stress)
}).T
display(reporting)
reporting.to_csv(OUTPUT/'reporting_totals.csv')

def stress_normalization(base,stress):
    niidiff=float(stress.nii.sum()-base.nii.sum())
    exposure=float((base.total*base.tau).sum())
    return {'Stress NII impact ($MM)':niidiff/1e6,
            'Normalized stress impact (annualized bps)':10000*niidiff/exposure,
            'Stress margin impact (reported bps)':report_summary(stress)['Reported margin (bps)']-report_summary(base)['Reported margin (bps)'],
            'Stress margin impact (annualized bps)':report_summary(stress)['Annualized margin (bps)']-report_summary(base)['Annualized margin (bps)']}
normalization=pd.DataFrame([stress_normalization(prior_base,prior_stress),
                            stress_normalization(current_base,current_stress)],index=['Prior','Current'])
display(normalization)
normalization.to_csv(OUTPUT/'normalized_stress.csv')


## 6. Time-path diagnostic (not a second reporting total)

The path decomposition changes period by period. **Cumulative NII difference** is the sum of period NII differences. The final aggregate margin is **recomputed from the entire selected period**, rather than summing period margin percentages.

In [ ]:
period_detail=pair_path['monthly'].copy()
period_detail['period_label']=[('PQ' if FREQUENCY=='Q' else 'M')+str(i) for i in period_detail['month_index']]
period_detail['difference']=period_detail['nii_b']-period_detail['nii_a']
period_detail['cumulative_difference']=period_detail['difference'].cumsum()
fig,ax=plt.subplots(figsize=(9,3.6))
ax.plot(period_detail['period_label'],period_detail['cumulative_difference']/1e6,marker='o',lw=2)
ax.axhline(0,color='gray',lw=.8)
ax.set(xlabel='Projection period',ylabel='Cumulative NII difference ($MM)',
       title='Outlook − CCAR Base: how the final NII difference accumulates')
ax.grid(alpha=.2)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(OUTPUT/'05_cumulative_nii_path.png',dpi=160)
plt.show()
period_detail.to_csv(OUTPUT/'period_path_nii.csv',index=False)


## 7. Reconciliation, aggregate-versus-path invariance and sensitivity checks

These tests independently confirm FTP-minus-expense accounting, both final reporting margins, both Shapley reconciliations, and cycle-over-cycle bridges. A uniform scaling of deposits and their dollar flows changes NII but not margin.

In [ ]:
for d in (a,b,prior_base,prior_stress,current_base,current_stress):
    assert np.allclose(d.nii,d.ftp-d.expense)
    assert np.allclose(d.total,d.dda+d.ib)
    direct=report_summary(d)
    assert np.isclose(d.nii.sum(),direct['Cumulative NII ($)'],atol=1e-3)
    assert np.isclose(direct['Reported margin (bps)'],
                      direct['Annualized margin (bps)']*direct['Horizon (years)'],atol=1e-7)

for d1,d2 in [(a,b),(prior_base,prior_stress),(current_base,current_stress)]:
    path=compare(d1,d2)
    agg=compare_aggregate(d1,d2)
    for key in ['nii_a','nii_b','margin_a','margin_b']:
        assert np.isclose(path[key],agg[key],atol=1e-3 if key.startswith('nii') else 1e-7)
    assert np.isclose(sum(path['n'].values()),path['nii_b']-path['nii_a'],atol=1e-3)
    assert np.isclose(sum(agg['n'].values()),agg['nii_b']-agg['nii_a'],atol=1e-3)
    assert np.isclose(sum(path['m'].values()),path['margin_b']-path['margin_a'],atol=1e-7)
    assert np.isclose(sum(agg['m'].values()),agg['margin_b']-agg['margin_a'],atol=1e-7)

assert np.isclose(sum(cycle_nii.values()),current_stress_nii-prior_stress_nii,atol=1e-3)
assert np.isclose(sum(cycle_margin.values()),current_stress_margin-prior_stress_margin,atol=1e-7)

scaled=b.copy()
for field in ('total','dda','ib','ftp','expense','nii'):
    scaled[field]*=1.25
scaled['mix']=scaled.dda/scaled.total
scaled['ftp_yield']=scaled.ftp/(scaled.total*scaled.tau)
scaled['paid_yield']=scaled.expense/(scaled.ib*scaled.tau)
assert np.isclose(report_summary(b)['Reported margin (bps)'],report_summary(scaled)['Reported margin (bps)'],atol=1e-7)
assert np.isclose(report_summary(b)['Annualized margin (bps)'],report_summary(scaled)['Annualized margin (bps)'],atol=1e-7)

orders=list(itertools.permutations(FACTORS))
assert len(orders)==24
print('PASS: all path, aggregate, margin convention, cycle bridge, and scaling checks.')
print('Results in:',OUTPUT.resolve())


### Interpretation and key limits

- **Both methods exactly explain the final P&R NII and margin:** total FTP minus total interest expense, divided by the average balance (reported), with an annualized alternative.
- **Path Shapley:** switches entire balance, DDA-share, blended-FTP-yield, and IB-paid-rate *series* and recomputes cumulative NII and full-horizon margin at every step. This preserves **balance timing effects** on the reported margin.
- **Aggregate Shapley:** first computes full-horizon average balance, DDA share, blended FTP yield, and IB paid yield from totals, then decomposes. Aggregate balance has **zero standalone contribution to margin** because aggregate margin cancels scale. The allocation differs from path Shapley, although final endpoints agree exactly.
- **Accounting caveat:** With only total FTP dollars, the split between FTP yield and DDA/IB mix is assumption-dependent; the mix contribution relates to IB paid expense, while FTP-related mix effects remain in the blended FTP yield.
- **Period 0 caveat:** Include PQ0/M0 only when its FTP and interest expense are genuine full-period flows; otherwise it is an initial snapshot.
- **Across cycles:** Path labels align projection offsets (PQ1 to PQ9), not necessarily calendar dates; compare cycles on matched projection horizons.


## 8. Optional: see why the two decompositions differ

The table below compares the **same endpoint** under two allocations. A nonzero difference between path and aggregate component values is **not an error**: path Shapley retains time-varying rates and volume-weighting, while aggregate Shapley compresses them. Both sets sum to the same total change. Use the aggregate version when explaining the reported end-of-horizon number succinctly; use path version when asking how monthly/quarterly balance **timing** contributed.

In [ ]:
difference=comparison[['Driver','Path NII ($MM)','Aggregate NII ($MM)']].copy()
difference['Path minus aggregate ($MM)']=difference['Path NII ($MM)']-difference['Aggregate NII ($MM)']
display(difference)
assert np.isclose(difference['Path minus aggregate ($MM)'].sum(),0,atol=1e-6)
print('Sum of allocation differences is zero: both explain the same cumulative NII.')